# 🌌 Results Preview: Massive Simulation Analysis

This notebook reads the raw distributed results (Parquet) from GCS and creates a 3D visualization of the photon point cloud around the black hole.

In [ ]:
# 1. Forced Headless Backend for Remote Dataproc Stability
%matplotlib inline
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import pandas as pd
from IPython.display import Image, display

print("Libraries loaded successfully.")

In [ ]:
# 2. Load Simulation Data from GCS using Spark
output_path = "gs://black-hole-visualizer-project-bh-vis-dataproc-config/sim_results/phase2_test"

try:
    # Attempt to read with Spark (Standard on Dataproc cluster)
    df_spark = spark.read.parquet(output_path)
    print(f"Successfully loaded {df_spark.count()} simulation points.")
    
    # Sample the data for visualization (plotting millions of points locally will crash)
    # We take a sample to keep it responsive
    df_pd = df_spark.sample(withReplacement=False, fraction=0.01).toPandas()
except Exception as e:
    print(f"Error: Ensure the simulation has been run and path exists: {e}")

In [ ]:
# 3. 3D Coordinate Transformation & Visualization
if 'df_pd' in locals():
    r = df_pd['r']
    theta = df_pd['theta']
    phi = df_pd['phi']

    # Boyer-Lindquist to Quasi-Cartesian
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)

    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection='3d')

    # Scatter plot of photon points
    # Color based on radius for visual depth
    scatter = ax.scatter(x, y, z, c=r, cmap='inferno', s=1, alpha=0.5)
    
    # Mark the Event Horizon (r=2M)
    u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
    ex = 2.0 * np.cos(u) * np.sin(v)
    ey = 2.0 * np.sin(u) * np.sin(v)
    ez = 2.0 * np.cos(v)
    ax.plot_wireframe(ex, ey, ez, color="black", alpha=0.2)

    ax.set_title("Massive Simulation Result: 3D Photon Visualization")
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    # Force consistent scale
    limit = 25
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_zlim(-limit, limit)

    # Save and display (Force stability on remote kernels)
    plt.savefig('result_preview.png')
    plt.close()
    display(Image('result_preview.png'))
else:
    print("No data available to plot. Run the simulation job first!")